# 阶段 2：特征工程

## 特征工程说明

基于阶段 1 清洗后的 SPY 行情数据，构建以下基础技术特征：

- **收益率特征**: return_1d（日收益率）、return_5d（5日收益率）
- **移动均线**: ma_5、ma_10、ma_20、ma_60
- **波动率**: volatility_20（20日滚动标准差）
- **成交量均线**: volume_ma_20
- **价格比率**: close_ma20_ratio（收盘价 / 20日均线）

同时保留原始 OHLCV 字段。

注意事项：
- 本阶段仅做特征构建和数据展示，不涉及预测、回测或交易策略
- 由于 rolling 和 pct_change 会产生 NaN，已删除包含 NaN 的行

In [ ]:
import pandas as pd
from pathlib import Path

# 探测项目根目录
cwd = Path.cwd()
if (cwd / "data").exists() and (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists() and (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "未找到项目根目录，请从项目根目录或 notebooks 目录启动 Notebook。"
    )

CLEAN_PATH = PROJECT_ROOT / "data" / "processed" / "SPY_clean_2015_2025.csv"

if CLEAN_PATH.exists():
    df_clean = pd.read_csv(CLEAN_PATH, index_col=0, parse_dates=True)
    print(f'清洗数据: {df_clean.shape[0]} 行, {df_clean.shape[1]} 列')
    print(f'日期范围: {df_clean.index.min().strftime("%Y-%m-%d")} ~ {df_clean.index.max().strftime("%Y-%m-%d")}')
    display(df_clean.head())
else:
    print(f'文件不存在: {CLEAN_PATH.resolve()}')
    print('请先运行: python run_stage1.py')


In [ ]:
# 读取特征数据
FEAT_PATH = PROJECT_ROOT / "data" / "features" / "SPY_features_2015_2025.csv"

if FEAT_PATH.exists():
    df_feat = pd.read_csv(FEAT_PATH, index_col=0, parse_dates=True)
    print(f'=== 特征数据 ===')
    print(f'shape: {df_feat.shape}')
    print(f'columns ({len(df_feat.columns)}):')
    for col in df_feat.columns:
        print(f'  {col}')
    print(f'\n=== 前5行 ===')
    display(df_feat.head())
    print(f'\n=== 后5行 ===')
    display(df_feat.tail())
else:
    print(f'文件不存在: {FEAT_PATH.resolve()}')
    print('请先运行: python run_stage2.py')


In [ ]:
# 读取特征摘要
SUMMARY_PATH = PROJECT_ROOT / "outputs" / "tables" / "SPY_feature_summary.csv"

if SUMMARY_PATH.exists():
    df_summary = pd.read_csv(SUMMARY_PATH)
    print(f'特征摘要共 {len(df_summary)} 项指标')
    print()
    # 基本信息
    basic_items = ['总行数', '总列数', '起始日期', '结束日期', '缺失值总数']
    for item in basic_items:
        mask = df_summary['指标'] == item
        if mask.any():
            print(f'{item}: {df_summary.loc[mask, "值"].values[0]}')
    print()
    # 显示完整摘要表
    display(df_summary)
else:
    print(f'文件不存在: {SUMMARY_PATH.resolve()}')
    print('请先运行: python run_stage2.py')


## 阶段 2 小结

- 基于阶段 1 的清洗数据，成功构建了 9 项基础技术特征
- 特征数据已保存至 `data/features/SPY_features_2015_2025.csv`
- 特征摘要已保存至 `outputs/tables/SPY_feature_summary.csv`
- 数据包含原始 OHLCV + 收益率、移动均线、波动率、成交量均线、价格比率
- 所有 NaN 行已通过 dropna() 清除，数据干净完整
- 本阶段仅完成特征构建，未涉及预测、回测或交易策略
- 特征数据可供后续实验阶段直接使用